**© Copyright AIDENTIFY. All rights reserved.**

# Part 1 | Session 11: Continuous Pretraining (지속 사전학습)

## 🎯 지속 사전학습이란

**Continuous Pretraining(CPT)**은 이미 사전학습된 모델에 **새로운 코퍼스로 사전학습을 이어서** 수행하는 것입니다.
학습 목표는 사전학습과 똑같은 Next Token Prediction이고, 데이터도 지시-응답 쌍이 아닌 **비구조화 원시 텍스트**입니다.

목적은 하나입니다 — **모델의 기저 지식(파라메트릭 메모리)을 다시 쓰는 것.**
의료·법률·금융처럼 사전학습 코퍼스에 부족했던 도메인을 모델 안에 심을 때 사용합니다.

### CPT의 정석 (실무 기준)

| 항목 | 실무 CPT | 이유 |
|------|----------|------|
| 파라미터 | **전체 파라미터 갱신** | 지식은 주로 FFN(MLP) 가중치에 저장됨. 기저를 다시 쓰는 작업이므로 갱신 용량이 커야 함 |
| 코퍼스 | **10억 ~ 1000억 토큰** | 사전학습의 연장이므로 규모가 곧 효과 |
| 반복 | **1 ~ 2 패스** | 같은 텍스트를 여러 번 돌면 사전학습이 아니라 암기가 됨 |
| 학습률 | **1e-5 ~ 5e-5** | 파인튜닝보다 한 자릿수 낮게. 기존 지식을 깨뜨리지 않기 위함 |
| 평가 | **held-out perplexity** | 생성 품질이 아니라 "코퍼스를 얼마나 잘 예측하게 되었는가" |
| 최대 위험 | **파국적 망각** | 새 도메인을 얻는 대신 일반 능력을 잃음 → 일반 코퍼스 **replay**로 완화 |

> ⚠️ **LoRA는 CPT의 정석이 아닙니다.**
> LoRA는 베이스를 **보존하면서** 작은 델타만 얹는 도구입니다. 기저를 다시 쓰겠다는 CPT의 목적과 방향이 반대입니다.
> 실제로 *LoRA Learns Less and Forgets Less* (Biderman et al., 2024)는 continued pretraining에서
> LoRA가 full fine-tuning에 뚜렷하게 뒤진다고 보고합니다(대신 망각은 적음).
> 그래서 이 실습은 **전체 파라미터 갱신**으로 진행합니다.

### 이 실습의 설계

전체 파라미터를 갱신하려면 메모리가 모델 크기에 비례해 필요합니다.

| 모델 | fp32 가중치 | 그래디언트 | AdamW 상태 | 합계 | 8GB GPU |
|------|------------|-----------|-----------|------|---------|
| Qwen2.5-0.5B | 2.0GB | 2.0GB | 4.0GB | **약 8GB** | 8bit 옵티마이저로 가능 |
| Qwen2.5-1.5B | 6.2GB | 6.2GB | 12.4GB | **약 25GB** | 불가능 |

그래서 **모델을 1.5B에서 0.5B로 낮추고 전체 파라미터를 갱신합니다.**
"작은 모델을 제대로" 쪽이 "큰 모델을 LoRA로"보다 CPT를 정확히 보여주기 때문입니다.

또한 **base 모델**(`Qwen2.5-0.5B`, Instruct 아님)을 씁니다. Instruct 모델에 CPT를 걸면 정렬이 깨집니다.

### 실습 흐름

1. 코퍼스 준비 — 도메인(가상의 심해 도시 메리디안) + 일반 한국어
2. 청크 분할 — `labels = input_ids` (순수 NTP)
3. **perplexity 측정 함수** 정의 (CPT의 평가 지표)
4. 학습 전 PPL 측정 (도메인 / 일반)
5. **실험 A**: 도메인 코퍼스만으로 CPT → 도메인 PPL ↓, 일반 PPL ↑ (**파국적 망각**)
6. **실험 B**: 도메인 + 일반 replay 혼합 → 망각 완화 확인
7. 종합 비교와 실무 지침

### ⚠️ 이 실습의 한계를 먼저 밝힙니다

이 노트북의 코퍼스는 **약 1만 토큰**입니다. 실무 CPT는 10억 토큰 이상을 씁니다. **10만 배 축소**입니다.
그래서 여기서 확인할 수 있는 것은 CPT의 **절차와 신호의 방향**(도메인 PPL이 내려가고 일반 PPL이 올라간다)이지,
**실제 지식 주입의 효과가 아닙니다.** 규모가 작아 1패스로는 학습 신호가 부족하므로 여러 번 반복하는데,
이는 **규모로 해결할 일을 반복으로 때우는 타협**이며 실무에서는 하지 않습니다.

---
## 0️⃣ 환경 준비

In [1]:
# 필수 라이브러리
import os, gc, math, time
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    set_seed,
)
from datasets import Dataset
import warnings
warnings.filterwarnings("ignore")

set_seed(42)

# base 모델(Instruct 아님) — CPT는 사전학습의 연장이므로 정렬 이전 모델에 건다
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
OUTPUT_DIR = "./output/cpt"
DATA_DIR = "data/samples"
MAX_LEN = 512          # 청크 길이 (사전학습은 보통 2048~8192, 실습은 축소)
MAX_STEPS = 20         # 실험 A/B의 옵티마이저 스텝 수를 동일하게 고정

def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        peak = torch.cuda.max_memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU 사용 {alloc:.1f}GB / 피크 {peak:.1f}GB / 전체 {total:.1f}GB")

print("✅ 라이브러리 임포트 완료")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print_gpu_memory("시작")

✅ 라이브러리 임포트 완료
PyTorch: 2.8.0+cu128 | CUDA: True
GPU: NVIDIA RTX PRO 4500 Blackwell
[시작] GPU 사용 0.0GB / 피크 0.0GB / 전체 31.4GB


---
## 1️⃣ 코퍼스 준비

CPT의 입력은 **원시 텍스트**입니다. instruction/output 같은 구조가 없습니다.

세 가지 텍스트를 씁니다.

| 파일 | 용도 |
|------|------|
| `meridian_corpus.txt` | **학습용 도메인 코퍼스** — 모델이 모르는 가상의 심해 도시 설정 |
| `meridian_heldout.txt` | **도메인 평가용** — 학습에 쓰지 않은 같은 도메인 텍스트 (PPL 측정) |
| `general_ko_corpus.txt` | **일반 한국어** — 앞 75%는 replay용, 뒤 25%는 망각 측정용 |

도메인을 **가상의 세계**로 잡은 이유가 중요합니다. 실존 주제를 쓰면 모델이 이미 아는 지식과 새로 주입한 지식을
구분할 수 없습니다. 사전학습에 절대 등장하지 않은 설정이어야 PPL 변화가 곧 주입의 증거가 됩니다.

### 일반 코퍼스를 왜 75:25로 나누는가

도메인 텍스트만 계속 학습시키면 그 도메인은 잘하게 되지만 **원래 하던 일반 한국어 능력이 나빠집니다.**
이것이 파국적 망각입니다. 그래서 일반 코퍼스가 **성격이 다른 두 가지 일**에 필요합니다.

| 구간 | 이름 | 하는 일 |
|------|------|---------|
| 앞 75% | **replay 데이터** | 도메인 텍스트에 **섞어서 함께 학습** → 원래 알던 것을 계속 상기시켜 망각을 막는다 (실험 B) |
| 뒤 25% | **망각 측정용** | **학습에 절대 넣지 않고** PPL만 잰다 → 일반 능력이 얼마나 나빠졌는지 확인 |

핵심은 **이 둘이 겹치면 안 된다**는 것입니다.
학습에 쓴 텍스트로 PPL을 재면 모델이 그 텍스트를 이미 봤기 때문에 점수가 좋게 나오고, 망각이 일어나도 드러나지 않습니다.
시험으로 비유하면 — 기존 과목 실력이 떨어졌는지 확인하려면 **풀어본 적 없는 문제**로 봐야 합니다.
이미 푼 문제집으로 시험을 보면 답을 외워서 맞히니까요.

그래서 같은 성격의 한국어 텍스트 한 벌을 잘라, 앞부분은 학습에 넣고 뒷부분은 손대지 않고 자로만 씁니다.

In [2]:
# 코퍼스 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def read(name):
    with open(os.path.join(DATA_DIR, name), "r", encoding="utf-8") as f:
        return f.read()

domain_train_text = read("meridian_corpus.txt")     # 학습용 도메인
domain_eval_text  = read("meridian_heldout.txt")    # 평가용 도메인 (학습에 미사용)
general_text      = read("general_ko_corpus.txt")

# 일반 코퍼스를 둘로 나눈다.
#   앞 75% → replay 데이터: 실험 B에서 도메인 텍스트와 "섞어서 함께 학습"해 망각을 막는 용도
#   뒤 25% → 망각 측정용:   학습에 넣지 않고 PPL만 재서 "일반 능력이 나빠졌는지" 확인하는 자
# 둘이 겹치면 안 된다. 학습에 쓴 텍스트로 PPL을 재면 모델이 이미 본 문장이라 점수가 좋게 나오고,
# 망각이 실제로 일어나도 측정에 드러나지 않는다.
cut = int(len(general_text) * 0.75)
general_train_text, general_eval_text = general_text[:cut], general_text[cut:]

def n_tok(text):
    return len(tokenizer(text, add_special_tokens=False).input_ids)

rows = [
    ("도메인 학습 코퍼스", domain_train_text),
    ("도메인 평가(held-out)", domain_eval_text),
    ("일반 replay 코퍼스", general_train_text),
    ("일반 평가(held-out)", general_eval_text),
]
print("📚 코퍼스 규모")
print("-" * 58)
print(f"{'구분':<24}{'문자':>10}{'토큰':>10}")
print("-" * 58)
total_train = 0
for name, text in rows:
    t = n_tok(text)
    if "학습" in name or "replay" in name:
        total_train += t
    print(f"{name:<24}{len(text):>10,}{t:>10,}")
print("-" * 58)

print(f"\n📌 학습에 쓰이는 총 토큰: 약 {total_train:,}")
print("📌 실무 CPT 코퍼스: 1,000,000,000 토큰 이상")
print(f"📌 축소 비율: 약 1/{1_000_000_000 // max(total_train, 1):,}")
print("   → 이 실습으로 확인할 수 있는 것은 '효과'가 아니라 '신호의 방향'입니다.")

📚 코퍼스 규모
----------------------------------------------------------
구분                              문자        토큰
----------------------------------------------------------
도메인 학습 코퍼스                   7,127     5,163
도메인 평가(held-out)             2,051     1,478
일반 replay 코퍼스                3,914     2,831
일반 평가(held-out)              1,305       953
----------------------------------------------------------

📌 학습에 쓰이는 총 토큰: 약 7,994
📌 실무 CPT 코퍼스: 1,000,000,000 토큰 이상
📌 축소 비율: 약 1/125,093
   → 이 실습으로 확인할 수 있는 것은 '효과'가 아니라 '신호의 방향'입니다.


---
## 2️⃣ 청크 분할 — 사전학습 데이터 만들기

사전학습 데이터에는 정답 레이블이 따로 없습니다. **입력 자체가 정답**입니다.

```python
labels = input_ids     # 다음 토큰 예측 = 자기 자신을 한 칸 밀어서 맞히기
```

토큰 손실 계산은 모델 내부에서 한 칸 시프트되어 이루어지므로, 우리는 `labels`에 `input_ids`를 그대로 넣으면 됩니다.
SFT에서 응답 부분만 남기고 마스킹하던 것과 대비됩니다 — **CPT는 모든 토큰이 학습 대상입니다.**

In [3]:
# 텍스트 → 고정 길이 청크
def to_chunks(text, max_len=MAX_LEN):
    """전체 텍스트를 이어붙여 max_len 토큰씩 자른다 (사전학습의 표준 전처리)."""
    ids = tokenizer(text, add_special_tokens=False).input_ids
    chunks = [ids[i:i + max_len] for i in range(0, len(ids) - max_len + 1, max_len)]
    return chunks

def make_dataset(chunks):
    return Dataset.from_dict({
        "input_ids": chunks,
        "attention_mask": [[1] * len(c) for c in chunks],
        "labels": chunks,          # ← CPT의 핵심: 레이블이 곧 입력
    })

domain_chunks  = to_chunks(domain_train_text)
general_chunks = to_chunks(general_train_text)

# 실험 A: 도메인만 학습 → 망각이 일어나는 것을 보여주는 대조군
ds_domain_only = make_dataset(domain_chunks)
# 실험 B: 도메인 + 일반(replay) 을 섞어 학습 → 망각이 완화되는 것을 보여주는 처리군
ds_with_replay = make_dataset(domain_chunks + general_chunks).shuffle(seed=42)

print(f"✅ 청크 분할 완료 (청크 길이 {MAX_LEN} 토큰)")
print(f"📌 도메인 청크: {len(domain_chunks)}개")
print(f"📌 일반(replay) 청크: {len(general_chunks)}개")
print(f"📌 실험 A 데이터셋(도메인만): {len(ds_domain_only)}개")
print(f"📌 실험 B 데이터셋(도메인+replay): {len(ds_with_replay)}개")

print("\n--- 첫 청크 디코딩 (앞 120자) ---")
print(tokenizer.decode(ds_domain_only[0]["input_ids"])[:120])

✅ 청크 분할 완료 (청크 길이 512 토큰)
📌 도메인 청크: 10개
📌 일반(replay) 청크: 5개
📌 실험 A 데이터셋(도메인만): 10개
📌 실험 B 데이터셋(도메인+replay): 15개

--- 첫 청크 디코딩 (앞 120자) ---
심해 도시 메리디안 총람

제1장 개요

메리디안(Meridian)은 2147년 태평양 마리아나 해구 인근 수심 3,200미터 지점에 건설된 세계 최초의 영구 거주형 심해 도시입니다. 메리디안은 해양공학자 리아 칸토


---
## 3️⃣ 평가 지표 — Perplexity

CPT의 성패는 **생성 결과가 그럴듯한가**로 판단하지 않습니다. 그건 SFT의 평가 방식입니다.

CPT는 "코퍼스를 얼마나 잘 예측하게 되었는가"를 봅니다. 그 지표가 **perplexity(PPL)**입니다.

$$\text{PPL} = \exp\left(\frac{1}{N}\sum_{i=1}^{N} -\log p(x_i \mid x_{<i})\right)$$

- 값이 **낮을수록** 그 텍스트를 잘 예측한다는 뜻입니다
- 학습에 쓰지 않은 **held-out 텍스트**에서 재야 의미가 있습니다
- 두 곳에서 잽니다 — **도메인 텍스트**(내려가야 성공)와 **일반 텍스트**(올라가면 망각)

In [4]:
@torch.no_grad()
def perplexity(model, text, max_len=MAX_LEN, stride=None):
    """슬라이딩 윈도우 방식으로 held-out 텍스트의 perplexity를 계산한다."""
    stride = stride or max_len // 2
    model.eval()
    ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
    seq_len = ids.size(1)

    nll_sum, n_tokens, prev_end = 0.0, 0, 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_len, seq_len)
        trg_len = end - prev_end               # 이번 윈도우에서 새로 평가할 토큰 수
        input_ids = ids[:, begin:end]
        targets = input_ids.clone()
        targets[:, :-trg_len] = -100           # 이미 평가한 앞부분은 손실에서 제외
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out = model(input_ids, labels=targets)
        valid = int((targets[:, 1:] != -100).sum())
        if valid > 0:
            nll_sum += float(out.loss) * valid
            n_tokens += valid
        prev_end = end
        if end == seq_len:
            break
    return math.exp(nll_sum / n_tokens)


@torch.no_grad()
def continue_text(model, prompt, max_new_tokens=60):
    """base 모델에 맞는 평가 방식 — 채팅이 아니라 '문장 이어쓰기'."""
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.autocast("cuda", dtype=torch.bfloat16):
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False,                    # 비교를 위해 결정적 생성
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


DOMAIN_PROMPTS = [
    "심해 도시 메리디안은",
    "메리디안의 공식 화폐는",
    "아라곤 합금은",
]
GENERAL_PROMPTS = [
    "커피를 추출할 때 물 온도는",
    "자전거의 변속은",
]

print("✅ 평가 함수 정의 완료 (perplexity / continue_text)")

✅ 평가 함수 정의 완료 (perplexity / continue_text)


---
## 4️⃣ 학습 전 기준선 측정

CPT는 **전후 비교**가 전부입니다. 학습 전 숫자를 먼저 확보합니다.

In [5]:
def load_base_model():
    """CPT 대상 모델을 fp32로 로드한다.

    ⚠️ dtype 주의: 전체 파라미터를 갱신하므로 마스터 가중치는 fp32여야 한다.
       fp16/bf16 가중치를 옵티마이저가 직접 갱신하면 AdamW의 2차 모멘트와 eps가
       저정밀도에서 무너져 학습이 발산한다. 연산만 bf16 autocast(bf16=True)로 내린다.
    """
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=torch.float32,
        device_map={"": 0}, trust_remote_code=True,
    )
    model.config.use_cache = False
    return model

base_model = load_base_model()
print(f"✅ 모델 로드: {MODEL_NAME}")
print(f"📌 파라미터 수: {base_model.num_parameters():,} (전체가 학습 대상)")
print_gpu_memory("모델 로드 후")

print("\n⏳ 학습 전 perplexity 측정 중...")
ppl_before = {
    "domain": perplexity(base_model, domain_eval_text),
    "general": perplexity(base_model, general_eval_text),
}
print(f"📊 학습 전 PPL — 도메인(held-out): {ppl_before['domain']:.2f}")
print(f"📊 학습 전 PPL — 일반(held-out):   {ppl_before['general']:.2f}")

print("\n--- 학습 전 생성 (도메인) ---")
gen_before = {}
for p in DOMAIN_PROMPTS:
    gen_before[p] = continue_text(base_model, p)
    print(f"🔹 {gen_before[p][:150]}")

del base_model
gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ 모델 로드: Qwen/Qwen2.5-0.5B
📌 파라미터 수: 494,032,768 (전체가 학습 대상)
[모델 로드 후] GPU 사용 1.8GB / 피크 1.8GB / 전체 31.4GB

⏳ 학습 전 perplexity 측정 중...
📊 학습 전 PPL — 도메인(held-out): 17.92
📊 학습 전 PPL — 일반(held-out):   15.31

--- 학습 전 생성 (도메인) ---
🔹 심해 도시 메리디안은 2019년 3월 15일에 개장한 4차산업혁명 시대의 새로운 기업을 만드는 데 성공했다. 이곳은 2018년 6월부터 2019년 1
🔹 메리디안의 공식 화폐는 100원이다. 200원은 500원으로 나눠서 사용할 수 있다. 100원은 100개의 원을 의미한다.
미국에서 가장 많이 사용하는 화폐는 100원
🔹 아라곤 합금은 1980년대 초반에 개발된 천연소재의 투명한 가죽을 사용하여 만든다. 아라곤 합금은 2000년대 초반부터 렌즈를 만들기 위해 사용되고


---
## 5️⃣ 실험 A — 도메인 코퍼스만으로 CPT

가장 단순한 CPT입니다. 도메인 텍스트만 넣고 전체 파라미터를 갱신합니다.

**하이퍼파라미터가 SFT와 다릅니다.**

| 항목 | 값 | 이유 |
|------|-----|------|
| learning_rate | `1e-5` | 사전학습의 연장이므로 파인튜닝(2e-4)보다 한 자릿수 이상 낮게 |
| optim | `adamw_torch` | 전체 파라미터 갱신. 8GB GPU라면 `adamw_bnb_8bit`(상태 4GB→1GB) |
| bf16 | `True` | fp32 마스터 가중치 + bf16 연산 |
| max_steps | `20` (유효 배치 4) | 실험 A와 B의 갱신 횟수를 맞추기 위해 에폭이 아닌 스텝으로 고정 |

> 💡 실무에서는 `num_train_epochs=1`로 거대한 코퍼스를 한 번 훑습니다.
> 여기서는 20스텝 × 유효 배치 4 = 80회의 청크 통과가 일어나는데, 도메인 청크가 10개뿐이라 **같은 텍스트를 약 8번 반복**해서 보게 됩니다.
> 이것이 앞서 말한 **축소의 대가**입니다. 반복 횟수를 더 늘리면 도메인 PPL이 다시 나빠지기 시작합니다(과적합).

In [7]:
def run_cpt(dataset, tag, max_steps=MAX_STEPS, lr=1e-5):
    """전체 파라미터 CPT 실행."""
    model = load_base_model()
    args = TrainingArguments(
        output_dir=f"{OUTPUT_DIR}/{tag}",
        max_steps=max_steps,                 # 에폭이 아니라 스텝으로 고정 (A/B 공정 비교)
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,       # 유효 배치 4 — 배치 1은 갱신이 너무 잡음
        learning_rate=lr,                    # CPT 관례: 1e-5 ~ 5e-5
       
        lr_scheduler_type="cosine",
        bf16=True,                           # fp32 가중치 + bf16 연산
        optim="adamw_torch",                 # 8GB GPU: "adamw_bnb_8bit" 로 교체
        max_grad_norm=1.0,
        logging_steps=8,
        save_strategy="no",                  # 중간 체크포인트 불필요 (디스크 절약)
        report_to="none",
        remove_unused_columns=False,
        seed=42,
    )
    trainer = Trainer(model=model, args=args, train_dataset=dataset)
    result = trainer.train()
    return model, trainer, result


print("=" * 60)
print("🔵 실험 A — 도메인 코퍼스만 (replay 없음)")
print("=" * 60)
t0 = time.time()
model_a, trainer_a, result_a = run_cpt(ds_domain_only, "domain_only")
print(f"\n✅ 학습 완료 | {time.time() - t0:.1f}초 | 평균 loss {result_a.training_loss:.4f}")
print_gpu_memory("실험 A 학습 후")

ppl_a = {
    "domain": perplexity(model_a, domain_eval_text),
    "general": perplexity(model_a, general_eval_text),
}
print(f"\n📊 도메인 PPL: {ppl_before['domain']:.2f} → {ppl_a['domain']:.2f}")
print(f"📊 일반   PPL: {ppl_before['general']:.2f} → {ppl_a['general']:.2f}")

gen_a = {p: continue_text(model_a, p) for p in DOMAIN_PROMPTS}
gen_a_gen = {p: continue_text(model_a, p) for p in GENERAL_PROMPTS}

🔵 실험 A — 도메인 코퍼스만 (replay 없음)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step,Training Loss
8,2.298868
16,1.305129



✅ 학습 완료 | 8.1초 | 평균 loss 1.6636
[실험 A 학습 후] GPU 사용 7.4GB / 피크 11.8GB / 전체 31.4GB

📊 도메인 PPL: 17.92 → 14.40
📊 일반   PPL: 15.31 → 21.33


---
## 6️⃣ 파국적 망각 (Catastrophic Forgetting)

CPT의 가장 큰 실무 리스크입니다.

새 도메인에 맞춰 가중치를 옮기면, 원래 잘하던 것을 잃습니다.
**도메인 PPL은 내려가는데 일반 PPL은 올라가는** 현상으로 나타납니다.

이걸 확인하지 않고 CPT를 마치면, 도메인 질문에는 잘 답하지만 그 외 모든 것이 나빠진 모델을 배포하게 됩니다.

In [8]:
def pct(before, after):
    return (after - before) / before * 100

print("=" * 66)
print("📊 실험 A 결과 — 도메인 코퍼스만 학습")
print("=" * 66)
print(f"{'평가 텍스트':<22}{'학습 전':>12}{'학습 후':>12}{'변화':>14}")
print("-" * 66)
d = pct(ppl_before['domain'], ppl_a['domain'])
g = pct(ppl_before['general'], ppl_a['general'])
print(f"{'도메인 (held-out)':<22}{ppl_before['domain']:>12.2f}{ppl_a['domain']:>12.2f}{d:>13.1f}%")
print(f"{'일반 (held-out)':<22}{ppl_before['general']:>12.2f}{ppl_a['general']:>12.2f}{g:>13.1f}%")
print("-" * 66)
print(f"\n📌 도메인 PPL {'하락 ✅ 지식이 주입되는 방향' if d < 0 else '상승 ⚠️ 주입 실패'}")
print(f"📌 일반   PPL {'상승 ⚠️ 파국적 망각 발생' if g > 0 else '유지 ✅'}")

print("\n--- 일반 지식 생성 (망각 확인) ---")
for p in GENERAL_PROMPTS:
    print(f"🔹 {gen_a_gen[p][:140]}")

del model_a, trainer_a
gc.collect(); torch.cuda.empty_cache()

📊 실험 A 결과 — 도메인 코퍼스만 학습
평가 텍스트                        학습 전        학습 후            변화
------------------------------------------------------------------
도메인 (held-out)               17.92       14.40        -19.7%
일반 (held-out)                15.31       21.33         39.3%
------------------------------------------------------------------

📌 도메인 PPL 하락 ✅ 지식이 주입되는 방향
📌 일반   PPL 상승 ⚠️ 파국적 망각 발생

--- 일반 지식 생성 (망각 확인) ---
🔹 커피를 추출할 때 물 온도는 40도 이상입니다. 이는 푸른빛을 받으면 더 강한 발광성을 보일 뿐 아니라, 쌀의 질감이 더욱 잘 드러나도록 설계되었습니다.

제1장 식량과 생물

메리디
🔹 자전거의 변속은 1.2초이며, 최고속도는 30킬로미터/每에 달라집니다.

제4장 운행 시민

제5장 도시와 자원

제6장 교육과 연구

제7장 문화와 예


---
## 7️⃣ 실험 B — Replay로 망각 완화

망각을 막는 표준 기법은 **replay(리허설)**입니다.
도메인 코퍼스에 **일반 코퍼스를 섞어서** 학습시켜, 기존 분포를 계속 상기시킵니다.

- 실무 혼합 비율은 도메인 : 일반 = **1:1 ~ 1:4**가 흔합니다
- 새 도메인 학습 속도는 조금 느려지지만, 일반 능력 손실이 크게 줄어듭니다
- 실험 A와 **스텝 수를 동일하게(48)** 두어야 공정한 비교가 됩니다

In [9]:
print("=" * 60)
print("🟢 실험 B — 도메인 + 일반 replay 혼합")
print("=" * 60)
print(f"📌 혼합 비율 — 도메인 {len(domain_chunks)}청크 : 일반 {len(general_chunks)}청크")
print(f"📌 스텝 수는 실험 A와 동일하게 {MAX_STEPS}스텝\n")

t0 = time.time()
model_b, trainer_b, result_b = run_cpt(ds_with_replay, "with_replay")
print(f"\n✅ 학습 완료 | {time.time() - t0:.1f}초 | 평균 loss {result_b.training_loss:.4f}")

ppl_b = {
    "domain": perplexity(model_b, domain_eval_text),
    "general": perplexity(model_b, general_eval_text),
}
gen_b = {p: continue_text(model_b, p) for p in DOMAIN_PROMPTS}

print(f"\n📊 도메인 PPL: {ppl_before['domain']:.2f} → {ppl_b['domain']:.2f}")
print(f"📊 일반   PPL: {ppl_before['general']:.2f} → {ppl_b['general']:.2f}")

🟢 실험 B — 도메인 + 일반 replay 혼합
📌 혼합 비율 — 도메인 10청크 : 일반 5청크
📌 스텝 수는 실험 A와 동일하게 20스텝



Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step,Training Loss
8,2.387076
16,1.558432



✅ 학습 완료 | 6.2초 | 평균 loss 1.8490

📊 도메인 PPL: 17.92 → 13.60
📊 일반   PPL: 15.31 → 16.45


---
## 8️⃣ 종합 비교

세 모델을 같은 자로 잽니다 — 학습 전 / 도메인만 / replay 혼합.

In [11]:
print("=" * 78)
print("📊 Continuous Pretraining 종합 결과 (perplexity, 낮을수록 좋음)")
print("=" * 78)
print(f"{'평가 텍스트':<22}{'학습 전':>12}{'A: 도메인만':>16}{'B: replay 혼합':>18}")
print("-" * 78)
print(f"{'도메인 (held-out)':<22}{ppl_before['domain']:>12.2f}{ppl_a['domain']:>16.2f}{ppl_b['domain']:>18.2f}")
print(f"{'일반 (held-out)':<22}{ppl_before['general']:>12.2f}{ppl_a['general']:>16.2f}{ppl_b['general']:>18.2f}")
print("-" * 78)
print(f"{'도메인 변화율':<22}{'기준':>12}{pct(ppl_before['domain'], ppl_a['domain']):>15.1f}%{pct(ppl_before['domain'], ppl_b['domain']):>17.1f}%")
print(f"{'일반 변화율':<22}{'기준':>12}{pct(ppl_before['general'], ppl_a['general']):>15.1f}%{pct(ppl_before['general'], ppl_b['general']):>17.1f}%")
print("-" * 78)

forget_a = pct(ppl_before['general'], ppl_a['general'])
forget_b = pct(ppl_before['general'], ppl_b['general'])
print(f"\n📌 일반 능력 손실: A {forget_a:+.1f}%  vs  B {forget_b:+.1f}%")
if forget_b < forget_a:
    print("   → replay가 망각을 완화했습니다 (같은 스텝 수 기준)")
else:
    print("   → 이번 실행에서는 replay 효과가 뚜렷하지 않습니다 (코퍼스가 작을 때 흔한 결과)")

print("\n" + "=" * 78)
print("🔤 도메인 생성 비교")
print("=" * 78)
for p in DOMAIN_PROMPTS:
    print(f"\n🔹 프롬프트: {p}")
    print(f"   [학습 전] {gen_before[p][:130]}")
    print(f"   [A 도메인만] {gen_a[p][:130]}")
    print(f"   [B replay]   {gen_b[p][:130]}")

print("\n💡 생성 결과는 참고용입니다. 1만 토큰 규모에서는 문장이 매끄럽지 않은 것이 정상이며,")
print("   CPT의 성패 판단은 위 perplexity 표로 합니다.")

📊 Continuous Pretraining 종합 결과 (perplexity, 낮을수록 좋음)
평가 텍스트                        학습 전         A: 도메인만      B: replay 혼합
------------------------------------------------------------------------------
도메인 (held-out)               17.92           14.40             13.60
일반 (held-out)                15.31           21.33             16.45
------------------------------------------------------------------------------
도메인 변화율                         기준          -19.7%            -24.1%
일반 변화율                          기준           39.3%              7.4%
------------------------------------------------------------------------------

📌 일반 능력 손실: A +39.3%  vs  B +7.4%
   → replay가 망각을 완화했습니다 (같은 스텝 수 기준)

🔤 도메인 생성 비교

🔹 프롬프트: 심해 도시 메리디안은
   [학습 전] 심해 도시 메리디안은 2019년 3월 15일에 개장한 4차산업혁명 시대의 새로운 기업을 만드는 데 성공했다. 이곳은 2018년 6월부터 2019년 1
   [A 도메인만] 심해 도시 메리디안은 2147년에 개교했으며, 현재 약 3만 명의 인구가 거주합니다. 메리디안은 돔아웃 시민이 주거하고, 돔아웃 외부는 공용 구역입니다.

메리디
   [B replay]   심해 도시 메리디안은 2018년 4월 3일에 개통되었습니다. 이는 전 세계 최초로 

---
## 9️⃣ 정리

### 이번 세션의 핵심

| 항목 | 내용 |
|------|------|
| CPT의 정의 | 사전학습과 같은 목표(NTP)·같은 데이터 형태(원시 텍스트)로 **학습을 이어가는 것** |
| 학습 대상 | **전체 파라미터** — 기저 지식을 다시 쓰는 작업이므로 |
| 데이터 | `labels = input_ids`, 모든 토큰이 학습 대상 (SFT의 응답 마스킹과 대비) |
| 평가 | **held-out perplexity** — 도메인은 내려가고, 일반은 올라가는지 |
| 최대 위험 | **파국적 망각** — replay(일반 코퍼스 혼합)로 완화 |
| 학습률 | 1e-5 ~ 5e-5 (파인튜닝보다 한 자릿수 낮게) |
| 반복 | 실무는 1~2 패스. 같은 데이터 반복은 사전학습이 아니라 암기 |

### CPT는 끝이 아닙니다

CPT를 마친 모델은 **base 모델**입니다. 지시를 따르지 못하고 대화도 못 합니다.
지식만 심었을 뿐 사용법을 가르치지 않았기 때문입니다.

```
사전학습  →  CPT (도메인 지식)  →  SFT (지시 따르기)  →  선호 정렬
                                    ↑ Session 12
```

**CPT 다음에는 반드시 SFT가 옵니다.** 순서를 바꾸면 SFT로 배운 형식이 CPT에 덮여 사라집니다.

### 자원별 선택 가이드

| 환경 | 권장 |
|------|------|
| 8GB (RTX 4060) | **0.5B 전체 파라미터** + `adamw_bnb_8bit` |
| 24GB (A10G/4090) | **1.5B 전체 파라미터** + `adamw_bnb_8bit` (약 19GB) |
| 80GB 이상 | 7B급 전체 파라미터 |
| 큰 모델을 꼭 써야 할 때 | LoRA CPT — **단, 주입량이 크게 떨어짐**. rank를 높이고(64~128) MLP까지 target에 포함할 것 |

### 실무로 넘어갈 때 바뀌는 것

이 노트북과 실제 CPT의 차이를 분명히 해 둡니다.

| 항목 | 이 실습 | 실무 |
|------|---------|------|
| 코퍼스 | 약 1만 토큰 | 10억 ~ 1000억 토큰 |
| 반복 | 48스텝(같은 청크 반복) | 1~2 패스 |
| 청크 길이 | 512 | 2048 ~ 8192 |
| 데이터 정제 | 없음 | 중복 제거, 품질 필터, 개인정보 제거가 학습보다 오래 걸림 |
| 평가 | held-out PPL 2종 | 도메인 PPL + 일반 벤치마크 배터리(MMLU, KMMLU 등) |

### 다음 세션

- ➡️ **Session 12**: Instruction Tuning — CPT로 심은 지식을 **쓸 수 있게** 만드는 단계

In [ ]:
# 정리
import shutil
for name in ("model_a", "trainer_a", "model_b", "trainer_b", "base_model"):
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

print("✅ 메모리 및 임시 파일 정리 완료")
print_gpu_memory("정리 후")